In [ ]:
# Configuration module - Window & Screen Settings

from dataclasses import dataclass
from typing import Optional
from PyQt6.QtWidgets import QApplication
from PyQt6.QtGui import QScreen

@dataclass
class ScreenInfo:
    """Store screen information"""
    name: str
    index: int
    width: int
    height: int
    pos_x: int
    pos_y: int
    phys_width_mm: float
    phys_height_mm: float
    dpi_x: float
    dpi_y: float
    dpi_avg: float
    diag_inches: float
    
    def print_info(self):
        """Pretty print screen information"""
        print(f"Monitor {self.index}: {self.name}")
        print(f"  Resolution: {self.width} x {self.height} px")
        print(f"  Position: ({self.pos_x}, {self.pos_y})")
        print(f"  Physical Size: {self.phys_width_mm:.0f} x {self.phys_height_mm:.0f} mm (~{self.diag_inches:.1f}\")")
        print(f"  DPI: {self.dpi_avg:.0f} (X: {self.dpi_x:.0f}, Y: {self.dpi_y:.0f})")

@dataclass
class WindowConfig:
    """Centralized window configuration"""
    title: str = "Wacom Pen Test - PyQt6"
    target_monitor: int = 1  # 1-indexed (1=second monitor, 0=primary)
    width: Optional[int] = None  # None = screen width
    height: Optional[int] = None  # None = screen height
    
class ScreenManager:
    """Manage screen detection and configuration"""
    
    @staticmethod
    def get_screen_info(screen: QScreen, index: int) -> ScreenInfo:
        """Extract and compute screen information"""
        geometry = screen.geometry()
        phys_size = screen.physicalSize()
        
        # Calculate actual DPI from physical size and resolution
        dpi_x = (geometry.width() * 25.4) / phys_size.width() if phys_size.width() > 0 else 0
        dpi_y = (geometry.height() * 25.4) / phys_size.height() if phys_size.height() > 0 else 0
        dpi_avg = (dpi_x + dpi_y) / 2
        
        # Calculate diagonal in inches
        diag_inches = (phys_size.width()**2 + phys_size.height()**2)**0.5 / 25.4
        
        return ScreenInfo(
            name=screen.name(),
            index=index,
            width=geometry.width(),
            height=geometry.height(),
            pos_x=geometry.x(),
            pos_y=geometry.y(),
            phys_width_mm=phys_size.width(),
            phys_height_mm=phys_size.height(),
            dpi_x=dpi_x,
            dpi_y=dpi_y,
            dpi_avg=dpi_avg,
            diag_inches=diag_inches
        )
    
    @staticmethod
    def get_all_screens(app: QApplication) -> list[ScreenInfo]:
        """Get info for all connected screens"""
        screens = []
        for i, screen in enumerate(app.screens(), 1):
            screens.append(ScreenManager.get_screen_info(screen, i))
        return screens
    
    @staticmethod
    def print_all_screens(screens: list[ScreenInfo]):
        """Print all screen information"""
        print(f"\n{'='*60}")
        print(f"📊 SCREEN INFORMATION")
        print(f"{'='*60}")
        print(f"Total screens detected: {len(screens)}\n")
        
        for screen in screens:
            screen.print_info()
            print()
    
    @staticmethod
    def get_target_screen(screens: list[ScreenInfo], config: WindowConfig) -> ScreenInfo:
        """Select target screen based on config"""
        if len(screens) > 1 and config.target_monitor > 1:
            target = screens[config.target_monitor - 1]
        else:
            target = screens[0]
        
        print(f"→ Opening on Monitor {target.index}: {target.name}\n")
        return target


In [ ]:
# use venv wacom_test 

import sys
from PyQt6.QtWidgets import QApplication, QWidget
from PyQt6.QtGui import QTabletEvent
from PyQt6.QtCore import Qt

class TabletTest(QWidget):
    def tabletEvent(self, event: QTabletEvent):
        # Print coordinates, pressure (0.0 - 1.0), and tilt angles
        print(f"X: {event.position().x():.1f}, Y: {event.position().y():.1f}, Pressure: {event.pressure():.2f}, Tilt X: {event.xTilt():.1f}°, Tilt Y: {event.yTilt():.1f}°")
        event.accept()

    def mousePressEvent(self, event):
        print(f"Mouse click at: X={event.position().x():.1f}, Y={event.position().y():.1f}")

# ===== CONFIGURATION =====
config = WindowConfig(
    title="Wacom Pen Test - PyQt6",
    target_monitor=2,  # 1=primary, 2=second monitor, etc.
)

# ===== SETUP =====
app = QApplication.instance()
if app is None:
    app = QApplication(sys.argv)

# Get screen information
screens = ScreenManager.get_all_screens(app)
ScreenManager.print_all_screens(screens)

# Get target screen
target_screen_info = ScreenManager.get_target_screen(screens, config)

# Create and configure window
w = TabletTest()
w.setWindowTitle(config.title)

# Position and size window
w.move(target_screen_info.pos_x, target_screen_info.pos_y)
window_width = config.width if config.width else target_screen_info.width
window_height = config.height if config.height else target_screen_info.height
w.resize(window_width, window_height)

# Show window
w.show()
w.raise_()
w.activateWindow()
w.setFocus()

print(f"{'='*60}\n")
app.exec()